# Patrón Estructural: Decorator

## Introducción
El patrón Decorator permite añadir funcionalidades a objetos colocando estos objetos dentro de objetos encapsuladores especiales que contienen estas funcionalidades.

## Objetivos
- Comprender cómo extender funcionalidades de objetos en tiempo de ejecución.
- Identificar cuándo es útil el patrón Decorator.
- Comparar la solución con y sin el patrón.

## Ejemplo de la vida real
**Contexto: App de Mensajería**
Supón que tienes un sistema de mensajes y quieres añadir funcionalidades como encriptación o compresión sin modificar la clase base. El patrón Decorator permite agregar estas funcionalidades de forma flexible.

**¿Dónde se usa en proyectos reales?**
En sistemas de mensajería, procesamiento de streams, frameworks de UI, etc.

## Sin patrón Decorator (forma errónea)
Se crean subclases para cada combinación de funcionalidades, lo que genera mucho código duplicado.

In [1]:
class Mensaje:
    def enviar(self, texto: str) -> None:
        print(f'Enviando: {texto}')

class MensajeEncriptado(Mensaje):
    def enviar(self, texto: str) -> None:
        print(f'Enviando encriptado: {texto}')

# Si agregas más funcionalidades, el número de clases crece exponencialmente

## Con patrón Decorator (forma correcta)
Se encapsulan funcionalidades adicionales en decoradores.

In [2]:
class Mensaje:
    def enviar(self, texto: str) -> None:
        print(f'Enviando: {texto}')

class DecoradorMensaje(Mensaje):
    def __init__(self, mensaje: Mensaje) -> None:
        self._mensaje = mensaje
    def enviar(self, texto: str) -> None:
        self._mensaje.enviar(texto)

class Encriptado(DecoradorMensaje):
    def enviar(self, texto: str) -> None:
        texto = f'***{texto}***'
        super().enviar(texto)

msg = Encriptado(Mensaje())
msg.enviar('Hola')

Enviando: ***Hola***


## UML del patrón Decorator
```plantuml
@startuml
class Mensaje {
    + enviar(texto)
}
class DecoradorMensaje {
    + enviar(texto)
}
Mensaje <|-- DecoradorMensaje
DecoradorMensaje <|-- Encriptado
DecoradorMensaje o-- Mensaje
@enduml
```

## Otro ejemplo de la vida real: Suscripciones SaaS con complementos (add-ons)
**Contexto:** herramientas como Notion o Slack cobran un plan base y permiten sumar complementos opcionales (soporte prioritario, almacenamiento extra, usuarios adicionales), cada uno agregando su propio costo al total. El número de combinaciones posibles crece rápido si cada combinación fuera una clase distinta.

### Sin patrón (forma errónea)
Una clase por cada combinación de plan + add-ons. Con solo 2 add-ons ya se necesitan 4 clases.

In [3]:
class PlanBasico:
    def costo(self) -> float:
        return 20

class PlanBasicoConSoporte:
    def costo(self) -> float:
        return 20 + 10

class PlanBasicoConAlmacenamiento:
    def costo(self) -> float:
        return 20 + 15

class PlanBasicoConSoporteYAlmacenamiento:
    def costo(self) -> float:
        return 20 + 10 + 15

# Cada nueva combinación de add-ons obliga a crear otra clase más

### Con patrón (forma correcta)
Cada add-on es un decorador que envuelve al plan y le suma su costo. Se pueden apilar tantos add-ons como se necesite, en cualquier orden, sin explosión de clases.

In [4]:
import abc

class Plan(abc.ABC):
    @abc.abstractmethod
    def costo(self) -> float:
        ...
    @abc.abstractmethod
    def descripcion(self) -> str:
        ...

class PlanBasico(Plan):
    def costo(self) -> float:
        return 20
    def descripcion(self) -> str:
        return 'Plan Básico'

class AddonPlan(Plan):
    def __init__(self, plan: Plan) -> None:
        self._plan = plan

class SoportePrioritario(AddonPlan):
    def costo(self) -> float:
        return self._plan.costo() + 10
    def descripcion(self) -> str:
        return self._plan.descripcion() + ' + Soporte prioritario'

class AlmacenamientoExtra(AddonPlan):
    def costo(self) -> float:
        return self._plan.costo() + 15
    def descripcion(self) -> str:
        return self._plan.descripcion() + ' + Almacenamiento extra'


plan = AlmacenamientoExtra(SoportePrioritario(PlanBasico()))
print(f'{plan.descripcion()}: ${plan.costo()}/mes')

Plan Básico + Soporte prioritario + Almacenamiento extra: $45/mes


### UML del ejemplo de suscripciones SaaS
```plantuml
@startuml
abstract class Plan {
    + costo()
    + descripcion()
}
class PlanBasico
abstract class AddonPlan {
    - _plan: Plan
}
class SoportePrioritario
class AlmacenamientoExtra
Plan <|-- PlanBasico
Plan <|-- AddonPlan
AddonPlan <|-- SoportePrioritario
AddonPlan <|-- AlmacenamientoExtra
AddonPlan o-- Plan
@enduml
```

### ¿Dónde más se usa Decorator?
- **Facturación SaaS con add-ons:** exactamente este ejemplo — Slack, Notion, AWS agregan costos de complementos sobre un plan/servicio base.
- **Streams de datos:** en Python, `io.TextIOWrapper`, `gzip.GzipFile` o `BufferedReader` decoran un stream base agregando codificación, compresión o buffering.
- **Middleware de frameworks web:** cada middleware (autenticación, logging, CORS) envuelve al siguiente, agregando comportamiento antes/después de manejar la petición.
- **UI con estilos apilables:** un componente visual al que se le agregan bordes, sombras o scroll, cada uno como una capa decoradora independiente.
- **Validación de datos:** decorar una función de validación base con reglas adicionales (longitud, formato, unicidad) que se pueden combinar libremente.

**Ejercicio de reflexión:** ¿el orden en que apilas los decoradores importa en este ejemplo (`AlmacenamientoExtra(SoportePrioritario(PlanBasico()))` vs. al revés)? Pruébalo y explica por qué sí o por qué no, pensando en qué pasaría si el costo de un add-on dependiera de un porcentaje en vez de un monto fijo.

## Actividad
Crea tu propio Decorator para agregar funcionalidades como compresión o registro de logs a un sistema de mensajes.

---
## Explicación de conceptos clave
- **Extensión flexible:** Permite agregar funcionalidades sin modificar la clase base.
- **Composición sobre herencia:** Favorece la composición de objetos en vez de la herencia múltiple.
- **Aplicación en la vida real:** Útil en sistemas de mensajería, procesamiento de streams y frameworks de UI.

## Conclusión
El patrón Decorator es ideal para añadir funcionalidades de forma flexible y escalable. Es común en sistemas de mensajería, procesamiento de datos y aplicaciones con múltiples opciones de extensión.